# SETUP

In [1]:
import io
import os       
import requests
import pandas as pd
import duckdb
import yfinance as yf
from tqdm.auto import tqdm
from dotenv import load_dotenv
from fredapi import Fred

# DATA INGESTION

In [2]:
PATH_MAIN = os.path.join(os.getcwd(), 'data')
PATH_HOLDINGS = f"{PATH_MAIN}/holdings"
PATH_PRICES = f"{PATH_MAIN}/prices"
PATH_FINANCIALS = f"{PATH_MAIN}/financials"
PATH_CALENDARS = f"{PATH_MAIN}/calendars"

os.makedirs(PATH_MAIN, exist_ok=True)
os.makedirs(PATH_HOLDINGS, exist_ok=True)
os.makedirs(PATH_PRICES, exist_ok=True)
os.makedirs(PATH_FINANCIALS, exist_ok=True)
os.makedirs(PATH_CALENDARS, exist_ok=True)

ETFS = [
  'XLC', 'XLY', 'XLP', 'XLE', 'XLF', 'XLV', 'XLI', 'XLB', 'XLRE', 'XAR',
  'KBE', 'XBI', 'KCE', 'XHE', 'XHS', 'XHB', 'KIE', 'XME', 'XES', 'XOP',
  'XPH', 'KRE', 'XRT', 'XSD', 'XSW', 'XTL', 'XTN', 'XLK', 'XLU'
]

ETFS_TEST = ['XLE', 'XLU']

## Download official SPDR holdings

In [3]:
def download_holdings(tickers, path=PATH_HOLDINGS):
    headers = {"User-Agent": "Mozilla/5.0"}
    
    for etf in tqdm(tickers):
        
        try:
            ssga_url = f'https://www.ssga.com/us/en/intermediary/library-content/products/fund-data/etfs/us/holdings-daily-us-en-{etf.lower()}.xlsx'
            response = requests.get(ssga_url, headers=headers)
            response.raise_for_status()
            
            # Save and partially clean individual files
            duckdb.sql(f"""
                copy (
                    select
                        '{etf}' as etf
                        , Name as name
                        , Ticker as ticker
                        , Weight::numeric as weight
                    from read_xlsx('{ssga_url}', range = 'A5:H')
                    where true
                        and SEDOL != '-'
                        and ticker != '-'
                ) to '{path}/{etf.lower()}_holdings.parquet'
            """)
            
        except Exception as e:
            print(f"{etf} error: {e}")
            continue

    # Save consolidated holdings file
    duckdb.sql(f"""
        copy (
            select *
            from read_parquet("{path}/*.parquet", union_by_name=True)
        ) to "{PATH_MAIN}/holdings.parquet"
    """)

In [4]:
# download_holdings(tickers=ETFS)

In [5]:
holdings = duckdb.sql(f"""
    select *
    from read_parquet("{PATH_MAIN}/holdings.parquet")
""").fetchdf()

In [6]:
holdings.info()

<class 'pandas.DataFrame'>
RangeIndex: 1768 entries, 0 to 1767
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   etf     1768 non-null   str    
 1   name    1768 non-null   str    
 2   ticker  1768 non-null   str    
 3   weight  1768 non-null   float64
dtypes: float64(1), str(3)
memory usage: 99.2 KB


In [7]:
holdings.head()

,etf,name,ticker,weight
0,KBE,BANCORP INC/THE,TBBK,1.115
1,KBE,JACKSON FINANCIAL INC A,JXN,1.059
2,KBE,ROCKET COS INC CLASS A,RKT,1.056
3,KBE,NICOLET BANKSHARES INC,NIC,1.053
4,KBE,NMI HOLDINGS INC,NMIH,1.048


## Download market data from yFinance

In [8]:
def download_prices(tickers, path=PATH_PRICES, interval="1d", period="5y"):
    unique_tickers = tickers.ticker.unique()
    print("Removed duplicate names.")
    unique_tickers = [ticker.replace('.', '-') for ticker in unique_tickers]
    print("Fixed ticker names.")

    for ticker in tqdm(unique_tickers):
        
        try:
            df = yf.download(
                ticker
                , interval=interval
                , period=period
                , multi_level_index=False
                , progress=False
                , auto_adjust=True
            ).reset_index()

            # Save and partially clean individual files
            duckdb.sql(f"""
                copy (
                    select
                        '{ticker}' as ticker
                        , Date::date as date
                        , Open as open
                        , High as high
                        , Low as low
                        , Close as close
                        , Volume as volume
                        , (close*volume)::bigint as value
                    from df
                ) to '{path}/{ticker.lower()}_prices.parquet'
            """)
            
        except Exception as e:
            print(f"{ticker} error: {e}")
            continue

    # Save consolidated prices file
    duckdb.sql(f"""
        copy (
            select *
            from read_parquet("{path}/*.parquet", union_by_name=True)
        ) to "{PATH_MAIN}/prices.parquet"
    """)    

In [9]:
# download_prices(holdings)

In [10]:
prices = duckdb.sql(f"""
    select *
    from read_parquet("{PATH_MAIN}/prices.parquet")
""").fetchdf()

In [11]:
prices.info()

<class 'pandas.DataFrame'>
RangeIndex: 1737391 entries, 0 to 1737390
Data columns (total 8 columns):
 #   Column  Dtype         
---  ------  -----         
 0   ticker  str           
 1   date    datetime64[us]
 2   open    float64       
 3   high    float64       
 4   low     float64       
 5   close   float64       
 6   volume  int64         
 7   value   int64         
dtypes: datetime64[us](1), float64(4), int64(2), str(1)
memory usage: 111.8 MB


In [12]:
prices.head()

,ticker,date,open,high,low,close,volume,value
0,A,2021-07-15,143.167524,144.103702,142.684958,143.765915,1733600,249232590
1,A,2021-07-16,144.123030,144.595950,143.099994,143.736969,2169800,311880475
2,A,2021-07-19,142.511233,142.810421,141.825984,142.434021,1835100,261380672
3,A,2021-07-20,143.273733,145.976104,142.675357,144.094101,2246000,323635351
4,A,2021-07-21,144.277435,144.682788,142.453338,143.756256,2255700,324270987


## Download earnings calendar from yFinance

In [13]:
def download_calendars(tickers, path=PATH_CALENDARS):
    unique_tickers = tickers.ticker.unique()
    print("Removed duplicate names.")
    unique_tickers = [ticker.replace('.', '-') for ticker in unique_tickers]
    print("Fixed ticker names.")
    
    for ticker in tqdm(unique_tickers):
        
        try:
            df = yf.Ticker(ticker).earnings_dates.reset_index()
        
            # Save and partially clean individual files
            duckdb.sql(f"""
                copy (
                    select
                        '{ticker}' as ticker
                        , "Earnings Date"::date as earnings_date
                        , "EPS Estimate" as eps_estimate
                        , "Reported EPS" as eps_actual
                        , "Surprise(%)" as surprise
                    from df
                ) to '{path}/{ticker.lower()}_calendars.parquet'
            """)
            
        except Exception as e:
            print(f"{ticker} error: {e}")
            continue

    # Save consolidated calendars file
    duckdb.sql(f"""
        copy (
            select *
            from read_parquet("{path}/*.parquet", union_by_name=True)
        ) to "{PATH_MAIN}/calendars.parquet"
    """)    

In [14]:
# download_calendars(holdings)

In [15]:
calendars = duckdb.sql(f"""
    select
        ticker
        , earnings_date
    from read_parquet("{PATH_MAIN}/calendars.parquet")
""").fetchdf()

In [16]:
calendars.info()

<class 'pandas.DataFrame'>
RangeIndex: 31967 entries, 0 to 31966
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   ticker         31967 non-null  str           
 1   earnings_date  31967 non-null  datetime64[us]
dtypes: datetime64[us](1), str(1)
memory usage: 607.6 KB


In [17]:
calendars.head()

,ticker,earnings_date
0,A,2026-08-27
1,A,2026-05-28
2,A,2026-02-26
3,A,2025-11-25
4,A,2025-08-28


## Download sentiment proxies from FRED

In [18]:
# load_dotenv()
# fred_api_key = os.getenv('fred')
# fred = Fred(api_key=fred_api_key)

# # CBOE Volatility Index: VIX (VIXCLS)
# vix = fred.get_series('VIXCLS')
# vix.name = 'vix'
# # ICE BofA AAA US Corporate Index Option-Adjusted Spread (BAMLC0A1CAAA)
# aaa = fred.get_series('BAMLC0A1CAAA')
# aaa.name = 'aaa'
# # ICE BofA BBB US Corporate Index Option-Adjusted Spread (BAMLC0A4CBBB)
# bbb = fred.get_series('BAMLC0A4CBBB')
# bbb.name = 'bbb'

In [19]:
# sentiment_proxies = pd.concat([vix, aaa, bbb], axis=1, sort=False).reset_index().to_parquet(f"{PATH_MAIN}/sentiment.parquet")

In [20]:
sentiment = duckdb.sql(f"""
    select
        index::date as date
        , vix
        , aaa
        , bbb
    from read_parquet("{PATH_MAIN}/sentiment.parquet")
""").fetchdf()

In [21]:
sentiment.info()

<class 'pandas.DataFrame'>
RangeIndex: 9543 entries, 0 to 9542
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   date    9543 non-null   datetime64[us]
 1   vix     9229 non-null   float64       
 2   aaa     786 non-null    float64       
 3   bbb     786 non-null    float64       
dtypes: datetime64[us](1), float64(3)
memory usage: 298.3 KB


In [22]:
sentiment.head()

,date,vix,aaa,bbb
0,1990-01-02,17.24,NaN,NaN
1,1990-01-03,18.19,NaN,NaN
2,1990-01-04,19.22,NaN,NaN
3,1990-01-05,20.11,NaN,NaN
4,1990-01-08,20.26,NaN,NaN


## Download official filings from EDGAR

### Download official EDGAR CIK list

In [23]:
def download_cik_lists():
    
    url = "https://www.sec.gov/files/company_tickers.json"
    headers = {"User-Agent": "Your Name your.email@example.com"}
    
    cik_list = requests.get(url, headers=headers).json()
    cik_list = pd.DataFrame(cik_list).T
    
    duckdb.sql(f"""
        copy (
            select
                'CIK' || lpad(cik_str::varchar, 10, '0') as cik
                , ticker
                , title as name
            from cik_list
        ) to '{PATH_MAIN}/ciks.parquet'
    """)
    
    print("Downloaded cik list.")
    
    mf_cik_list = "https://www.sec.gov/files/company_tickers_mf.json"
    mf_cik_list = requests.get(mf_cik_list, headers=headers).json()
    mf_cik_list_columns = mf_cik_list["fields"]
    mf_cik_list = pd.DataFrame(mf_cik_list["data"], columns=mf_cik_list_columns)
    
    duckdb.sql(f"""
        copy (
            select *
            from mf_cik_list
            where symbol in {ETFS}
        ) to '{PATH_MAIN}/ciks_etfs.parquet'
    """)
    
    print("Downloaded cik_etfs list.")

In [24]:
# download_cik_lists()

In [25]:
ciks = duckdb.sql(f"""
    select *
    from read_parquet("{PATH_MAIN}/ciks.parquet")
""").fetchdf()

In [26]:
ciks.info()

<class 'pandas.DataFrame'>
RangeIndex: 10428 entries, 0 to 10427
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   cik     10428 non-null  str  
 1   ticker  10428 non-null  str  
 2   name    10428 non-null  str  
dtypes: str(3)
memory usage: 650.0 KB


In [27]:
holdings_extended = duckdb.sql(f"""
    select
        etf
        , cik
        , a.ticker as ticker
        , a.name as name
        , weight
    from holdings a
    join ciks b on a.ticker == b.ticker
""").fetchdf()

In [28]:
holdings_extended.info()

<class 'pandas.DataFrame'>
RangeIndex: 1758 entries, 0 to 1757
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   etf     1758 non-null   str    
 1   cik     1758 non-null   str    
 2   ticker  1758 non-null   str    
 3   name    1758 non-null   str    
 4   weight  1758 non-null   float64
dtypes: float64(1), str(4)
memory usage: 134.7 KB


In [29]:
holdings_extended.head()

,etf,cik,ticker,name,weight
0,XSD,CIK0001045810,NVDA,NVIDIA CORP,2.530
1,XLK,CIK0000320193,AAPL,APPLE INC,12.536
2,XLC,CIK0001652044,GOOGL,ALPHABET INC CL A,10.746
3,XSW,CIK0000789019,MSFT,MICROSOFT CORP,0.682
4,XRT,CIK0001018724,AMZN,AMAZON.COM INC,1.403


### Scan relevant EDGAR API keys

In [121]:
# What I need --> valuation related --> price to sales/earnings

In [122]:
c = "CIK0001326801"
url = f"https://data.sec.gov/api/xbrl/companyfacts/{c}.json"
headers = {"User-Agent": "Mozilla/5.0"}
response = requests.get(url, headers=headers)
data = response.json()
all_gaap = data["facts"]["us-gaap"]
all_gaap_keys = all_gaap.keys()

In [123]:
print(f"Data keys: {list(data.keys())}")
print(f"Facts keys: {list(data["facts"].keys())}")

Data keys: ['cik', 'entityName', 'facts']
Facts keys: ['dei', 'us-gaap', 'srt', 'ecd', 'ffd']


In [124]:
pd.DataFrame(data["facts"]["us-gaap"]["EarningsPerShareBasic"]["units"]["USD/shares"]).sort_values("filed", ascending=False).head()

,start,end,val,accn,fy,fp,form,filed,frame
178,2025-01-01,2025-03-31,6.59,0001628280-26-028526,2026,Q1,10-Q,2026-04-30,CY2025Q1
184,2026-01-01,2026-03-31,10.57,0001628280-26-028526,2026,Q1,10-Q,2026-04-30,CY2026Q1
164,2023-01-01,2023-12-31,15.19,0001628280-26-003942,2025,FY,10-K,2026-01-29,CY2023
176,2024-01-01,2024-12-31,24.61,0001628280-26-003942,2025,FY,10-K,2026-01-29,CY2024
183,2025-01-01,2025-12-31,23.98,0001628280-26-003942,2025,FY,10-K,2026-01-29,CY2025


In [128]:
shares_keys = [c for c in all_gaap_keys if 'pershare' in c.lower()]
shares_keys.sort(key=len)
shares_keys[:3]

['EarningsPerShareBasic',
 'EarningsPerShareDiluted',
 'CommonStockParOrStatedValuePerShare']

In [129]:
revenue_keys = [c for c in all_gaap_keys if 'revenue' in c.lower()]
revenue_keys.sort(key=len)
revenue_keys[:3]

['Revenues', 'CostOfRevenue', 'AdvertisingRevenue']

In [130]:
income_keys = [c for c in all_gaap_keys if 'incomeloss' in c.lower()]
income_keys.sort(key=len)
income_keys[:3]

['NetIncomeLoss',
 'OperatingIncomeLoss',
 'NetIncomeLossAttributableToParentDiluted']

### Check financial data format

In [37]:
def download_shares(tickers, path=PATH_FINANCIALS):

    for _, row in tqdm(tickers[["ticker", "cik"]].iterrows()):
        ticker = row["ticker"]
        cik = row["cik"]

        url = f"https://data.sec.gov/api/xbrl/companyfacts/{cik}.json"
        headers = {"User-Agent": "Mozilla/5.0"}
        response = requests.get(url, headers=headers)
        response.raise_for_status()
        company_facts = response.json()
        
        try:
            shares = company_facts["facts"]["us-gaap"]["EarningsPerShareBasic"]["units"]["USD/shares"]
            df_shares = pd.DataFrame(shares)
            duckdb.sql(f"""copy (select '{ticker}' as ticker, 'shares' as item, * from df_shares) to '{path}/{ticker.lower()}_shares.parquet'""")
            
        except Exception as e:
            print(f"{ticker} error: {e}")

    # Save consolidated financials file
    duckdb.sql(f"""copy (select * from read_parquet('{path}/*shares.parquet', union_by_name=True)) to '{PATH_MAIN}/shares.parquet'""")

In [38]:
holdings_extended.cik.count()

np.int64(1758)

In [39]:
# download_shares(holdings_extended)

In [40]:
def download_financials(tickers, path=PATH_FINANCIALS):

    for _, row in tqdm(tickers[["ticker", "cik"]].iterrows()):
        ticker = row["ticker"]
        cik = row["cik"]

        url = f"https://data.sec.gov/api/xbrl/companyfacts/{cik}.json"
        headers = {"User-Agent": "Mozilla/5.0"}
        response = requests.get(url, headers=headers)
        response.raise_for_status()
        company_facts = response.json()
        
        try:
            shares = company_facts["facts"]["us-gaap"]["CommonStockSharesOutstanding"]["units"]["shares"]
            df_shares = pd.DataFrame(shares)
            duckdb.sql(f"""copy (select '{ticker}' as ticker, 'shares' as item, * from df_shares) to '{path}/{ticker.lower()}_shares.parquet'""")
            
        except Exception as e:
            print(f"{ticker} error: {e}")

        try:
            revenues = company_facts["facts"]["us-gaap"]["Revenues"]["units"]["USD"]
            df_revenues = pd.DataFrame(revenues)
            duckdb.sql(f"""copy (select '{ticker}' as ticker, 'revenues' as item, * from df_revenues) to '{path}/{ticker.lower()}_revenues.parquet'""")
            
        except Exception as e:
            print(f"{ticker} error: {e}")

        try:
            income = company_facts["facts"]["us-gaap"]["NetIncomeLoss"]["units"]["USD"]
            df_income = pd.DataFrame(income)
            duckdb.sql(f"""copy (select '{ticker}' as ticker, 'income' as item, * from df_income) to '{path}/{ticker.lower()}_income.parquet'""")
            
        except Exception as e:
            print(f"{ticker} error: {e}")
    
    # Save consolidated financials file
    duckdb.sql(f"""copy (select * from read_parquet('{path}/*shares.parquet', union_by_name=True)) to '{PATH_MAIN}/shares.parquet'""")    
    duckdb.sql(f"""copy (select * from read_parquet('{path}/*revenues.parquet', union_by_name=True)) to '{PATH_MAIN}/revenues.parquet'""")    
    duckdb.sql(f"""copy (select * from read_parquet('{path}/*income.parquet', union_by_name=True)) to '{PATH_MAIN}/income.parquet'""")    

In [41]:
# download_financials(holdings_extended)

## Confirm label concept using EDGAR Tools API

In [165]:
from edgar import *
set_identity("John Doe john.doe@company.com")

In [223]:
company = Company('OXY')
financials = company.get_financials().income_statement()
financials = financials.to_dataframe()
print(f'Industry: {company.industry}')
financials

Industry: Crude Petroleum & Natural Gas


,concept,label,standard_concept,2025-12-31 (FY),2024-12-31 (FY),2023-12-31 (FY),level,abstract,dimension,is_breakdown,dimension_axis,dimension_member,dimension_member_label,dimension_label,balance,weight,preferred_sign,parent_concept,parent_abstract_concept
0,oxy_RevenuesAndOtherIncomeAbstract,REVENUES AND OTHER INCOME,NaN,NaN,NaN,NaN,1,True,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,us-gaap_IncomeStatementAbstract
1,us-gaap_Revenues,Net sales,Revenue,2.159300e+10,2.201900e+10,2.315600e+10,2,False,False,False,NaN,NaN,NaN,NaN,credit,1.0,1.0,oxy_RevenuesOperatingAndNonoperating,oxy_RevenuesAndOtherIncomeAbstract
2,us-gaap_Revenues,Related Party,NaN,1.280000e+08,1.460000e+08,1.530000e+08,3,False,True,False,us-gaap:RelatedPartyTransactionsByRelatedParty...,us-gaap_RelatedPartyMember,Related Party,us-gaap:RelatedPartyTransactionsByRelatedParty...,credit,1.0,1.0,oxy_RevenuesOperatingAndNonoperating,oxy_RevenuesAndOtherIncomeAbstract
3,us-gaap_Revenues,Other Operating Segments And Intersegment Elim...,NaN,2.090200e+10,2.170500e+10,2.128400e+10,3,False,True,True,srt:ConsolidationItemsAxis,oxy_OtherOperatingSegmentsAndIntersegmentElimi...,Other Operating Segments And Intersegment Elim...,srt:ConsolidationItemsAxis: Other Operating Se...,credit,1.0,1.0,oxy_RevenuesOperatingAndNonoperating,oxy_RevenuesAndOtherIncomeAbstract
4,us-gaap_Revenues,Other Operating Segments And Intersegment Elim...,NaN,1.279000e+09,8.860000e+08,2.433000e+09,3,False,True,True,srt:ConsolidationItemsAxis,oxy_OtherOperatingSegmentsAndIntersegmentElimi...,Other Operating Segments And Intersegment Elim...,srt:ConsolidationItemsAxis: Other Operating Se...,credit,1.0,1.0,oxy_RevenuesOperatingAndNonoperating,oxy_RevenuesAndOtherIncomeAbstract
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
63,us-gaap_EarningsPerShareBasic,Net income attributable to common stockholders...,NaN,1.650000e+00,2.590000e+00,4.220000e+00,2,False,False,False,NaN,NaN,NaN,NaN,NaN,1.0,1.0,NaN,us-gaap_EarningsPerShareBasicAbstract
64,us-gaap_EarningsPerShareDilutedAbstract,"PER COMMON SHARE, DILUTED",NaN,NaN,NaN,NaN,1,True,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,us-gaap_IncomeStatementAbstract
65,us-gaap_IncomeLossFromContinuingOperationsPerD...,Income from continuing operations—diluted (in ...,NaN,1.350000e+00,2.230000e+00,2.490000e+00,3,False,False,False,NaN,NaN,NaN,NaN,NaN,1.0,1.0,us-gaap_EarningsPerShareDiluted,us-gaap_EarningsPerShareDilutedAbstract
66,us-gaap_DiscontinuedOperationIncomeLossFromDis...,Discontinued operations—diluted (in dollars pe...,NaN,2.600000e-01,2.100000e-01,1.410000e+00,3,False,False,False,NaN,NaN,NaN,NaN,NaN,1.0,1.0,us-gaap_EarningsPerShareDiluted,us-gaap_EarningsPerShareDilutedAbstract


# DATA PROCESSING TEMPLATES

In [ ]:
df_shares = 
duckdb.sql(f"""
    select 
        '{t}' as ticker
        , 'Shares' as item
        , form
        , filed::date as date
        , "end"::date as period_end
        , val::bigint as shares
    from df_shares
    where (filed, "end") in (
        select filed, max("end")
        from df_shares
        group by filed
    )
    order by filed desc
""").show()

In [ ]:
ticker = 'NVDA'
duckdb.sql(f"""
    with
    prep as (
        select
            '{ticker}' as ticker
            , 'Revenues' as item
            , form
            , filed::date as date
            , start::date as period_start
            , "end"::date as period_end
            , (julian("end"::date) - julian("start"::date))::int as days
            , val::bigint as ytd_revenues
        from read_parquet('{PATH_FINANCIALS}/{ticker.lower()}_revenues.parquet')
    ),
    qtr as (
        select *
            , case when days < 100 then ytd_revenues
                else ytd_revenues - lag(ytd_revenues, 1) over (order by period_end asc)
                end as qtr
        from prep
        where (date, period_end, days) in (
            select date, max(period_end), max(days)
            from prep group by date
        )
    ),
    ttm as (
    select *
        , lag(qtr, 4) over (order by period_end asc) as prev_qtr
        , sum(qtr) over (order by period_end asc
            rows between 3 preceding and current row) as ttm
    from qtr
    where qtr is not null
    ),
    chg as (
    select *
        , 100 * ((qtr / prev_qtr) - 1)::numeric as yoy_chg
        , 100 * ((ttm / lag(ttm, 1) over (order by period_end asc)) - 1)::numeric as ttm_chg
    from ttm
    )
    select ticker, item, form, date, qtr, ttm, yoy_chg, ttm_chg
    from chg
    where yoy_chg is not null
    order by date desc
""").show()

In [ ]:
duckdb.sql(f"""
    with
    prep as (
        select
            '{t}' as ticker
            , 'Income' as item
            , form
            , filed::date as date
            , start::date as period_start
            , "end"::date as period_end
            , (julian("end"::date) - julian("start"::date))::int as days
            , val::bigint as ytd_income
        from df_income
        where form in ('10-K', '10-Q')
    ),
    qtr as (
        select *
            , case when days < 100 then ytd_income
                else ytd_income - lag(ytd_income, 1) over (order by period_end asc)
                end as qtr
        from prep
        where (date, period_end, days) in (
            select date, max(period_end), max(days)
            from prep group by date
        )
    ),
    ttm as (
    select *
        , lag(qtr, 4) over (order by period_end asc) as prev_qtr
        , sum(qtr) over (order by period_end asc
            rows between 3 preceding and current row) as ttm
    from qtr
    where qtr is not null
    ),
    chg as (
    select *
        , 100 * ((qtr / prev_qtr) - 1)::numeric as yoy_chg
        , 100 * ((ttm / lag(ttm, 1) over (order by period_end asc)) - 1)::numeric as ttm_chg
    from ttm
    )
    select ticker, item, form, date, qtr, ttm, yoy_chg, ttm_chg
    from chg
    where yoy_chg is not null
    order by date desc
""").show()

In [ ]:
from edgar import *

set_identity("John Doe john.doe@company.com")

In [ ]:
# Get Apple's latest 10-K filing and parse XBRL
company = Company("NVDA")
filing = company.get_filings(form="10-K").latest()
xbrl = filing.xbrl()

# Get income statement with dimensional segment data
income_stmt = xbrl.statements.income_statement()

# Convert to DataFrame - includes product/service breakdowns by default
df = income_stmt.to_dataframe()

In [ ]:
print(df.info())
df[df.is_breakdown == True]